In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [24]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/Herald/6th Sem AI/Data/Workshop-8/trum_tweet_sentiment_analysis.csv')

# Display the first 5 rows and check columns
print(df.head())
print(df.info())

                                                text  Sentiment
0  RT @JohnLeguizamo: #trump not draining swamp b...          0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2  Trump protests: LGBTQ rally in New York https:...          1
3  "Hi I'm Piers Morgan. David Beckham is awful b...          0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1850123 entries, 0 to 1850122
Data columns (total 2 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   text       object
 1   Sentiment  int64 
dtypes: int64(1), object(1)
memory usage: 28.2+ MB
None


### Applying Text Cleaning to the Dataset

In [25]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [26]:
!pip install emoji
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
import emoji

def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This function cleans text data by lowercasing, removing URLs, emojis,
  punctuation, special characters, stopwords, and then tokenizes and
  applies either lemmatization or stemming.

  Args:
    dataset (str): The input text string to be cleaned.
    rule (str, optional): The rule for tokenization. Can be 'lemmatize' or 'stem'.
                          Defaults to 'lemmatize'.

  Returns:
    str: The cleaned and processed text.
  """
  # Convert the input to small/lower order.
  data = dataset.lower()

  # Remove URLs
  data = re.sub(r'http\S+|www\S+|https\S+', '', data, flags=re.MULTILINE)

  # Remove emojis
  data = emoji.demojize(data) # Converts emojis to text (e.g., :smile:)
  data = re.sub(r':\S+:', '', data) # Removes the demojized text

  # Remove mentions (@username)
  data = re.sub(r'@\w+', '', data)

  # Remove all other unwanted characters (punctuation, non-alphanumeric, etc.).
  # Keep only letters and spaces
  data = re.sub(r'[^a-z\s]', '', data)

  # Remove extra spaces
  data = re.sub(r'\s+', ' ', data).strip()

  # Create tokens.
  tokens = data.split()

  # Remove stopwords:
  stop_words = set(stopwords.words('english'))
  tokens = [word for word in tokens if word not in stop_words]

  if rule == "lemmatize":
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)

In [27]:
# Apply the text cleaning pipeline to the 'text' column
df['cleaned_text'] = df['text'].apply(text_cleaning_pipeline)

# Display the first few rows with the cleaned text
print(df[['text', 'cleaned_text', 'Sentiment']].head())

                                                text  \
0  RT @JohnLeguizamo: #trump not draining swamp b...   
1  ICYMI: Hackers Rig FM Radio Stations To Play A...   
2  Trump protests: LGBTQ rally in New York https:...   
3  "Hi I'm Piers Morgan. David Beckham is awful b...   
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...   

                                        cleaned_text  Sentiment  
0  rt trump draining swamp taxpayer dollar trip a...          0  
1  icymi hacker rig fm radio station play antitru...          0  
2    trump protest lgbtq rally new york bbcworld via          1  
3  hi im pier morgan david beckham awful donald t...          0  
4  rt tech firm suing buzzfeed publishing unverif...          0  


### Train-Test Split

In [28]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 1480098
Test set size: 370025


### TF-IDF Vectorization

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for efficiency

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

Shape of X_train_tfidf: (1480098, 5000)
Shape of X_test_tfidf: (370025, 5000)


### Model Training and Evaluation (Logistic Regression)

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize and train the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence
logistic_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = logistic_model.predict(X_test_tfidf)

# Print the classification report
print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred))

Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94    248842
           1       0.90      0.86      0.88    121183

    accuracy                           0.92    370025
   macro avg       0.92      0.91      0.91    370025
weighted avg       0.92      0.92      0.92    370025

